In [2]:
from agents import Agent, trace, Runner, WebSearchTool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, EmailStr, Field
from dotenv import load_dotenv, find_dotenv
import asyncio
import os

In [ ]:
load_dotenv(override = True)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")




In [ ]:
#Planner agent that findsout relevant keywords
class WebSearchItem(BaseModel):
    query : str = Field(description = "The term that needs to be searched.")
    reason: str = Field(description = " The reasoning why the query/keyword is relevant to the topic. ")

class WebSearchPlan(BaseModel):
    searches : list[WebSearchItem] = Field(description = " The list of search terms and reasonings that needs to be searched ")

async def search_planner(query : str):
    """ It plans the  search by finding a number of relevant keywords or terms along with reasons. """
    NUMBER_OF_SEARCHES = 3
    INSTRUCTIONS = f""" You  are a search planner who takes query and identifies {NUMBER_OF_SEARCHES} of terms relevant to query. \
        You come up with the queries along with reasoning why do you think those terms are relevant. While writing reasoning, be concise and write one liners."""
    query = query
    planner_agent = Agent(name = "Planner_Agent", model = "gpt-4o-mini", instructions = INSTRUCTIONS, output_type= WebSearchPlan)
    
    result = await Runner.run(planner_agent, query)
    return result.final_output


to_search = "ANU WHS Framework"
query = f"""query : {to_search}"""
print (await search_planner(query))



In [41]:
#Searching strategy item by item

async def search(query_reason : WebSearchItem):
    """ It is used for searching the information on the query provided. """
    INSTRUCTIONS = f""" You are a searcher whose job is to search most relevant information using the query and reason provided.\
         You are not concerned with grammer or completion of sentences. \
         Try to come up with the most relevant information in less than 300 words. """
    message = f""" Query to be searched is : {query_reason.query} \n The reason of the query is : {query_reason.reason} """
    search_agent = Agent(name="Search_Agent", model = "gpt-4o-mini", instructions = INSTRUCTIONS)
    result = await Runner.run(search_agent, message)
    result = result.final_output
    return result

#test = WebSearchItem(query='ANU Workplace Health and Safety policies', reason='To understand the specific policies governing WHS at Australian National University.')
#await search(test)



In [50]:
async def search_process(searches : WebSearchPlan):
    """ It searches each term one by one and compiles the results in the form of a list object. """
    tasks = []
    searches = searches
    #print ("searches look like this : ", searches)
    for search_ in searches.searches:
        task = asyncio.create_task(search(search_))
        tasks.append(task)
    results = await asyncio.gather(*tasks)
    print (results)
    return results

to_search = "ANU WHS Framework"
query = f"""query : {to_search}"""
searches = await search_planner(query)
searching = await search_process(searches)

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: OPENAI_A*************************************************************************************************************************************************************************YdoA. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: OPENAI_A*************************************************************************************************************************************************************************YdoA. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}


['ANU (Australian National University) has several key Work Health and Safety (WHS) policies designed to ensure a safe working environment. \n\n1. **WHS Policy**: Outlines commitment to maintaining health and safety, risk management approach, and responsibilities of staff and students.\n\n2. **WHS Management System**: Framework detailing processes for identifying, assessing, and controlling risks. It includes procedures for reporting incidents, conducting audits, and continuous improvement.\n\n3. **Risk Management Procedures**: Guidelines for conducting risk assessments, including identifying hazards, evaluating risks, and implementing controls. \n\n4. **Incident Reporting and Investigation**: Procedures for reporting workplace incidents, with forms and processes for investigation and follow-up actions.\n\n5. **Safety Training**: Mandatory training programs for staff and students, covering safe work practices, emergency procedures, and specific training for high-risk activities.\n\n6. 

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: OPENAI_A*************************************************************************************************************************************************************************YdoA. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}


In [ ]:
print(searching)



['ANU (Australian National University) has several key Work Health and Safety (WHS) policies designed to ensure a safe working environment. \n\n1. **WHS Policy**: Outlines commitment to maintaining health and safety, risk management approach, and responsibilities of staff and students.\n\n2. **WHS Management System**: Framework detailing processes for identifying, assessing, and controlling risks. It includes procedures for reporting incidents, conducting audits, and continuous improvement.\n\n3. **Risk Management Procedures**: Guidelines for conducting risk assessments, including identifying hazards, evaluating risks, and implementing controls. \n\n4. **Incident Reporting and Investigation**: Procedures for reporting workplace incidents, with forms and processes for investigation and follow-up actions.\n\n5. **Safety Training**: Mandatory training programs for staff and students, covering safe work practices, emergency procedures, and specific training for high-risk activities.\n\n6. 

In [56]:
#Report Writer

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")

async def report_writer(query : str, search_results : list):
    message = f""" Search term for writing report : {query} \
                Sources information to be used for writing reports ; {search_results} """
    INSTRUCTIONS = f""" You are a senior researcher tasked with writing a cohesive report for a research query. 
                    You will be provided with the original query, and some initial research done by a research assistant.\n
                    You should first come up with an outline for the report that describes the structure and 
                    flow of the report. Then, generate the report and return that as your final output.\n
                    The final output should be in markdown format, and it should be lengthy and detailed. Aim 
                    for 5-10 pages of content, at least 1000 words. """

    writer_agent = Agent(name = "Writer_Agent", model = "gpt-4o-mini", instructions = INSTRUCTIONS, output_type = ReportData)
    report_ = await Runner.run(writer_agent , message)
    report_ = report_.final_output
    return report_


In [60]:
to_search = "ANU's WHS Policy"
async def main(to_search : str):
    with trace("Mazhar Search Engine"):
        to_search = to_search
        query = f"""query : {to_search}"""
        searches = await search_planner(query)
        searching = await search_process(searches)
        report_writing = await report_writer(to_search , searching)
        return report_writing
    
    

In [61]:
final_result = await main(to_search)

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: OPENAI_A*************************************************************************************************************************************************************************YdoA. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: OPENAI_A*************************************************************************************************************************************************************************YdoA. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}


['ANU (Australian National University) Work Health and Safety (WHS) Policy details are outlined in the following key areas:\n\n1. **Purpose**: Ensure a safe and healthy environment for all staff, students, and visitors.\n\n2. **Scope**: Applies to all ANU campuses and activities, including physical spaces, events, and any work-related context.\n\n3. **Compliance**: Adheres to national WHS legislation and regulations to mitigate risks and obligations.\n\n4. **Responsibilities**:\n   - **Management**: Must ensure resources for health and safety, risk assessments, and incident reporting.\n   - **Workers**: Required to observe safety protocols and report hazards.\n   - **Health, Safety and Wellbeing Team**: Offers support for policy implementation and training.\n\n5. **Risk Management**: Policies include processes for identifying, assessing, and controlling risks. Regular reviews and audits are conducted.\n\n6. **Training and Awareness**: Mandatory training programs and resources for staff

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: OPENAI_A*************************************************************************************************************************************************************************YdoA. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}


[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: OPENAI_A*************************************************************************************************************************************************************************YdoA. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}


In [62]:
print(final_result)

short_summary='The ANU Work Health and Safety (WHS) Policy establishes a comprehensive framework for ensuring a safe and healthy environment for all individuals on campus. It details responsibilities for management and workers, risk management practices, mandatory training programs, and procedures for incident reporting and emergencies, aimed at fostering a culture of safety and compliance with national regulations.' markdown_report="# ANU Work Health and Safety (WHS) Policy Report\n\n## Table of Contents\n\n1. [Introduction](#introduction)\n2. [Purpose of the WHS Policy](#purpose-of-the-whs-policy)\n3. [Scope of the Policy](#scope-of-the-policy)\n4. [Compliance and Legal Framework](#compliance-and-legal-framework)\n5. [Roles and Responsibilities](#roles-and-responsibilities)\n   - 5.1 [Management Responsibilities](#management-responsibilities)\n   - 5.2 [Worker Responsibilities](#worker-responsibilities)\n   - 5.3 [Health, Safety and Wellbeing Team](#health-safety-and-wellbeing-team)\